# Практика 1. Transformer Encoder с нуля

## 0. Подготовка

Нужны только PyTorch и Matplotlib. Обозначения во всём ноутбуке:

- `B` — batch size;
- `T` — длина последовательности;
- `D` — размерность эмбеддинга (`d_model`);
- `H` — число attention heads;
- `Dh = D / H` — размерность одной головы.


In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. Embedding и позиционная информация

Self-attention сам по себе не знает порядок токенов. Добавим синусоидальное позиционное кодирование:

$$
PE_{pos,2i}=\sin(pos/10000^{2i/D}),\qquad
PE_{pos,2i+1}=\cos(pos/10000^{2i/D}).
$$


In [ ]:
def sinusoidal_positions(max_len: int, d_model: int) -> torch.Tensor:
    positions = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    frequencies = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10_000.0) / d_model)
    )
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(positions * frequencies)
    pe[:, 1::2] = torch.cos(positions * frequencies)
    return pe

pe = sinusoidal_positions(max_len=32, d_model=16)
print(pe.shape)
plt.figure(figsize=(8, 3))
plt.imshow(pe.T, aspect="auto", cmap="coolwarm")
plt.xlabel("position")
plt.ylabel("embedding coordinate")
plt.colorbar()
plt.show()


**Вопросы для обсуждения**

1. Почему одного token embedding недостаточно?
2. Чем обучаемые positional embeddings BERT отличаются от синусоидальных?
3. Что произойдёт, если переставить токены, но не переставлять positional embeddings?


## 2. Scaled dot-product attention

$$
Attention(Q,K,V)=softmax(\frac{QK^T}{\sqrt{D_h}}+M)V.
$$

Маска имеет форму, совместимую с `(..., T_query, T_key)`. Значение `False` запрещает смотреть на соответствующий key-токен. До `softmax` запрещённые logits заменяются на очень большое отрицательное число.


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    # q, k, v: (..., T, Dh); mask: broadcastable bool tensor.
    # TODO 1: вычислите scores = QK^T / sqrt(Dh)
    # TODO 2: примените bool-маску до softmax
    # TODO 3: получите weights и output
    raise NotImplementedError


q = torch.randn(2, 4, 8)
k = torch.randn(2, 4, 8)
v = torch.randn(2, 4, 8)
padding_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.bool)
mask = padding_mask[:, None, :]  # (B, 1, T_key)

out, weights = scaled_dot_product_attention(q, k, v, mask)
assert out.shape == (2, 4, 8)
assert weights.shape == (2, 4, 4)
assert torch.allclose(weights.sum(-1), torch.ones(2, 4), atol=1e-6)
assert weights[0, :, 3].max() < 1e-6
print("tests passed; output:", out.shape, "attention:", weights.shape)


### Почему делим на `sqrt(Dh)`?

Сравним энтропию attention при росте размерности. Без scaling скалярные произведения становятся крупнее, `softmax` — слишком «острым», а градиенты — менее информативными.


In [ ]:
def mean_attention_entropy(d_head, scaled, repeats=200):
    entropies = []
    for _ in range(repeats):
        q = torch.randn(1, 8, d_head)
        k = torch.randn(1, 8, d_head)
        scores = q @ k.transpose(-2, -1)
        if scaled:
            scores = scores / math.sqrt(d_head)
        p = scores.softmax(-1)
        entropies.append((-(p * (p + 1e-9).log()).sum(-1).mean()).item())
    return sum(entropies) / len(entropies)

for d in [4, 16, 64, 256]:
    raw = mean_attention_entropy(d, scaled=False)
    scaled = mean_attention_entropy(d, scaled=True)
    print(f"Dh={d:3d} | without scale={raw:.3f} | with scale={scaled:.3f}")


## 3. Multi-head self-attention

Каждая голова получает собственные проекции `Q`, `K`, `V`. Головы работают параллельно, затем их результаты конкатенируются и проходят выходную линейную проекцию.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None, return_attention=False):
        b, t, d = x.shape
        # TODO 1: одной проекцией получите Q, K, V.
        # TODO 2: приведите каждый tensor к форме (B, H, T, Dh).
        # TODO 3: преобразуйте padding_mask из (B,T) в (B,1,1,T).
        # TODO 4: вызовите scaled_dot_product_attention.
        # TODO 5: склейте головы обратно в (B,T,D) и примените self.out.
        raise NotImplementedError


x = torch.randn(2, 5, 32)
mha = MultiHeadSelfAttention(d_model=32, n_heads=4)
y, attn = mha(x, padding_mask=torch.ones(2, 5), return_attention=True)
assert y.shape == x.shape
assert attn.shape == (2, 4, 5, 5)
print("x:", x.shape, "y:", y.shape, "attention:", attn.shape)


## 4. Encoder-блок в стиле BERT

BERT использует **post-norm**: сначала подслой, затем residual connection и LayerNorm.

$$
h_1=LN(x+MHA(x)),\qquad h_2=LN(h_1+FFN(h_1)).
$$

`FFN` применяется к каждому токену независимо и обычно расширяет скрытую размерность примерно в 4 раза.


In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout)
        # TODO 1: соберите FFN: Linear -> GELU -> Dropout -> Linear.
        self.ffn = None
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None, return_attention=False):
        # TODO 2: реализуйте post-norm блок из формулы выше.
        raise NotImplementedError


block = TransformerEncoderBlock(d_model=32, n_heads=4, d_ff=128)
y, attn = block(torch.randn(2, 7, 32), return_attention=True)
assert y.shape == (2, 7, 32)
assert attn.shape == (2, 4, 7, 7)
print("encoder output:", y.shape)


## 5. Мини-BERT

Соберите masked language model для последовательностей вида `a, a+1, a+2, ...` по модулю 20.

Требования:

1. `Embedding` токенов и позиций;
2. два реализованных выше encoder-блока;
3. линейная MLM-голова размера `D → vocab_size`;
4. один случайный `[MASK]` на последовательность;
5. `CrossEntropyLoss(ignore_index=-100)` — loss только на скрытой позиции.


In [ ]:
PAD_ID, VOCAB_VALUES, MASK_ID = 0, 20, 21
VOCAB_SIZE, SEQ_LEN = 22, 8

def make_batch(batch_size):
    starts = torch.randint(1, VOCAB_VALUES + 1, (batch_size, 1))
    offsets = torch.arange(SEQ_LEN).unsqueeze(0)
    target = ((starts - 1 + offsets) % VOCAB_VALUES) + 1
    positions = torch.randint(0, SEQ_LEN, (batch_size,))
    inputs = target.clone()
    inputs[torch.arange(batch_size), positions] = MASK_ID
    labels = torch.full_like(target, -100)
    labels[torch.arange(batch_size), positions] = target[torch.arange(batch_size), positions]
    return inputs.to(device), labels.to(device), positions.to(device)

sample_x, sample_y, sample_pos = make_batch(3)
print("input:\n", sample_x.cpu())
print("labels (-100 ignored):\n", sample_y.cpu())


In [ ]:
class TinyMaskedLM(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=48, n_heads=4, d_ff=128, n_layers=2):
        super().__init__()
        # TODO: token embedding, position embedding, ModuleList encoder-блоков, lm_head.
        raise NotImplementedError

    def forward(self, input_ids, padding_mask=None):
        # TODO: сложите два типа embeddings, примените блоки и lm_head.
        raise NotImplementedError

model = TinyMaskedLM(VOCAB_SIZE, SEQ_LEN).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)


In [ ]:
# TODO: обучите модель 250 шагов. Loss считается по logits формы
# (B*T, VOCAB_SIZE) и labels формы (B*T,) с ignore_index=-100.
losses = []


In [ ]:
# TODO: посчитайте accuracy только на замаскированной позиции.
